In [ ]:
# GPU HARD GATE
import sys
import torch
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU is not available. Enable GPU accelerator before running.'
print('cuda device count:', torch.cuda.device_count())
print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
# USER PARAMETERS: edit this cell for single-case or full-test 4-label inference.
CHECKPOINT = '/kaggle/input/route-b-4label-full-train/best_checkpoint.pt'
MODE = 'test'
CASE_ID = None
BATCH_SIZE = 16
NUM_WORKERS = 2
USE_AMP = True
USE_DATA_PARALLEL = True
GPU_IDS = '0,1'
OUTPUT_DIR = 'run_outputs/route_b_4label_test'
SAVE_NIFTI_MASKS = True
SAVE_PROBABILITY_MAPS = False
SAVE_CASE_METRICS = True
SEED = 731
EXPORT_RESULTS = True
EXPORT_ROOT = '/kaggle/working/exports'
CODE_SRC = '/kaggle/input/datasets/mathewzhou/pbl-brain-tumour-code'
DATA_SRC = '/kaggle/input/datasets/mathewzhou/pbl-brain-tumour-training'


In [ ]:
# Setup a clean Kaggle working copy of the code dataset.
import os
import shutil
import tarfile
from pathlib import Path

CODE_SRC = Path(CODE_SRC)
PROJECT = Path('/kaggle/working/PBL_BrainTumour')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True, exist_ok=True)

def safe_extract_tar(tar_path, destination):
    destination = Path(destination).resolve()
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise RuntimeError(f'Unsafe tar member: {member.name}') from exc
        tar.extractall(destination)

if (CODE_SRC / 'src').exists() and (CODE_SRC / 'scripts').exists():
    for item in CODE_SRC.iterdir():
        if item.name.startswith('._') or item.name == '__MACOSX':
            continue
        dst = PROJECT / item.name
        if item.is_dir():
            shutil.copytree(item, dst, ignore=shutil.ignore_patterns('._*', '__MACOSX'))
        else:
            shutil.copy2(item, dst)
elif (CODE_SRC / 'kaggle_code.tar.gz').exists():
    safe_extract_tar(CODE_SRC / 'kaggle_code.tar.gz', PROJECT)
else:
    raise FileNotFoundError(f'Could not find expanded code or kaggle_code.tar.gz in {CODE_SRC}')
os.chdir(PROJECT)
print('cwd:', os.getcwd())
print('project files:', sorted(p.name for p in PROJECT.iterdir()))


In [ ]:
# Create data symlink expected by inference defaults.
DATA_SRC = Path(DATA_SRC)
DATA_LINK = PROJECT / 'data/processed/Task01_BrainTumour'
DATA_LINK.parent.mkdir(parents=True, exist_ok=True)
if DATA_LINK.exists() or DATA_LINK.is_symlink():
    if DATA_LINK.is_symlink() or DATA_LINK.is_file():
        DATA_LINK.unlink()
    else:
        shutil.rmtree(DATA_LINK)
DATA_LINK.symlink_to(DATA_SRC, target_is_directory=True)
print('data link:', DATA_LINK, '->', DATA_LINK.resolve())
print('checkpoint:', CHECKPOINT)
assert Path(CHECKPOINT).exists(), f'Checkpoint not found: {CHECKPOINT}'


In [ ]:
# Dependency, GPU, and required-path gate.
import importlib.util
import subprocess
missing = [pkg for pkg in ['timm', 'safetensors', 'nibabel', 'yaml'] if importlib.util.find_spec(pkg) is None]
print('missing packages:', missing)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'safetensors', 'nibabel', 'pyyaml'], check=True)
required = [
    PROJECT / 'data/processed/Task01_BrainTumour/imagesTr',
    PROJECT / 'src/pretrained_hybrid_vit.py',
    PROJECT / 'scripts/07_infer_route_b_4label.py',
    PROJECT / 'outputs/splits/frozen_patient_split_seed42.csv',
]
if MODE == 'test':
    required.append(PROJECT / 'data/processed/Task01_BrainTumour/labelsTr')
missing_paths = [str(path) for path in required if not path.exists()]
assert not missing_paths, (
    'Missing required paths:\n'
    + '\n'.join(missing_paths)
)
print('Environment gate passed.')


In [ ]:
# 4-label inference launcher.
def run_command(cmd):
    print('RUN:', ' '.join(str(x) for x in cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
cmd = [sys.executable, 'scripts/07_infer_route_b_4label.py', '--checkpoint', CHECKPOINT, '--mode', MODE, '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS), '--output-dir', OUTPUT_DIR, '--seed', str(SEED), '--gpu-ids', str(GPU_IDS)]
if CASE_ID is not None:
    cmd += ['--case-id', str(CASE_ID)]
if USE_AMP:
    cmd.append('--amp')
if USE_DATA_PARALLEL:
    cmd.append('--data-parallel')
if SAVE_NIFTI_MASKS:
    cmd.append('--save-nifti')
if SAVE_PROBABILITY_MAPS:
    cmd.append('--save-probability-maps')
if SAVE_CASE_METRICS:
    cmd.append('--save-case-metrics')
run_command(cmd)


In [ ]:
# Display and export inference outputs.
import pandas as pd
import os
run_dir = PROJECT / OUTPUT_DIR
metrics_path = run_dir / 'test_case_metrics.csv'
summary_path = run_dir / 'test_summary.json'
if metrics_path.exists():
    display(pd.read_csv(metrics_path))
if summary_path.exists():
    print(summary_path.read_text())
if EXPORT_RESULTS:
    EXPORT_ROOT = Path(EXPORT_ROOT)
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(EXPORT_ROOT / run_dir.name), 'gztar', root_dir=run_dir.parent, base_dir=run_dir.name)
    print('Export:', archive)
    print(f'Size: {os.path.getsize(archive) / 1024**2:.2f} MB')
